# Feature Engineering

This notebook is the fifth step in the data preparation pipeline, following EDA, missing-value treatment, outlier removal, and distribution transformations. Its purpose is to **create new features** from the existing columns, a process known as **feature engineering**.

**Why feature engineering matters:** The raw columns in a dataset rarely capture the full predictive signal available. By combining, transforming, or extracting information from existing features, we can create new variables that encode domain knowledge and expose patterns the model would otherwise struggle to learn on its own. For example, knowing the year an apartment was built is useful, but knowing its *age* (current year minus year built) is often more directly related to rent because depreciation and maintenance needs accumulate over time.

The new features we create fall into three categories:
1. **Extracted features** — pulling structured information out of unstructured or composite columns.
2. **Aggregate features** — combining multiple columns into a single, more informative one.
3. **Interaction features** — multiplying two features together to capture joint effects (e.g., size per room).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import date


In [2]:
df = pd.read_csv('spitogatos_transforms.csv')
df.head()


,listingId,airConditioning,alarm,areaSqm,balcony,bathrooms,closedParkingSpots,elevator,energyClass,fireplace,...,pool,price,pricePerSqm,propertyType,renovationYear,rooms,solarHeater,storage,view,yearBuilt
0,20304396,True,False,3.850148,False,1.0,0.0,True,C,False,...,False,6.396930,12.77,apartment,2026.0,1.0,False,False,True,1975.0
1,19956518,True,False,3.218876,False,1.0,0.0,True,B,False,...,False,6.272877,21.20,apartment,2023.0,1.0,False,False,True,1982.0
2,18383741,False,False,4.094345,False,1.0,0.0,True,D,False,...,False,6.234411,8.50,apartment,NaN,1.0,False,False,True,1978.0
3,20242029,True,False,4.094345,False,1.0,0.0,True,F,False,...,False,5.799093,5.50,apartment,NaN,1.0,False,False,True,1978.0
4,20055845,True,False,4.317488,False,1.0,0.0,True,C,False,...,False,6.429719,8.27,apartment,NaN,2.0,False,False,True,1981.0


## Extracted & Aggregate Features

### Extracting the Municipal District (`locationFloor`)

The raw `location` column contains a composite string like `"Antigonidon (Centre of Thessaloniki)"`. During EDA we identified that the text inside the parentheses represents the **municipal district** (city floor). We extract it here using a simple string-split operation, producing a new `locationFloor` column with 12 unique district names.

### Total Parking Spots (`totalParkingSpots`)

The dataset has two separate parking columns: `closedParkingSpots` (garage) and `openParkingSpots` (open-air). For modelling purposes, what usually matters is the *total* number of parking spots available. By summing the two, we create a single `totalParkingSpots` feature that captures the full parking capacity.

### Apartment Age (`apartmentAge`)

Rather than using `yearBuilt` directly, we compute `apartmentAge = current_year - yearBuilt`. This is a more intuitive feature because rent tends to decrease as a building ages (due to wear, outdated fixtures, etc.), and a linear model can more easily learn a negative relationship with "age" than with the raw construction year (where the relationship would be reversed). We use 2026 as the reference year.


In [3]:
df["locationFloor"] = df["location"].apply(lambda x: x.split("(")[1].split(")")[0].strip())
df['totalParkingSpots'] = df['closedParkingSpots'] + df['openParkingSpots']
current_year = 2026
df['apartmentAge'] = current_year - df['yearBuilt']


## Encoding the Municipal District (`district`)

While the 12 `locationFloor` categories are an improvement over 64 raw neighbourhoods, they are still too many for some models to handle efficiently (especially linear models). We further aggregate them into **5 municipal districts** using a manual mapping dictionary. The district numbers (1--5) reflect a rough ordering from central to peripheral:

- **District 1** — City centre and immediately adjacent areas (highest demand and rents).
- **Districts 2--5** — Progressively more peripheral areas.

This numeric encoding allows the model to treat district as an ordinal or one-hot encoded feature, depending on the algorithm chosen. The `.map()` function replaces each `locationFloor` string with its corresponding district number.


In [4]:
location_to_district = {
    "Κέντρο Θεσσαλονίκης": 1,
    "Φάληρο - Ιπποκράτειο": 1,
    "Τούμπα": 4,
    "Ανάληψη - Μπότσαρη - Νέα Παραλία": 5,
    "Θεσσαλονίκης - Δήμος": 1,
    "Βούλγαρη - Ντεπώ - Μαρτίου": 5,
    "Βαρδάρης - Λαχανόκηποι": 1,
    "Ξηροκρήνη - Παναγία Φανερωμένη": 2,
    "Τριανδρία - Δόξα - Κρυονέρι": 4,
    "Άνω Πόλη": 1,
    "40 Εκκλησίες - Ευαγγελιστρίας": 3,
    "Χαριλάου": 5
}

df['district'] = df['locationFloor'].map(location_to_district)


## Interaction & Indicator Features

### Renovation Indicator (`is_renovated`)

During the missing-values notebook we discovered that whether or not a `renovationYear` is recorded carries predictive information (apartments with no recorded renovation year tend to have higher average rents). We now formalise this finding as a **binary indicator feature**: `is_renovated = 1` if the renovation year is known, `0` otherwise. This allows the model to capture the "missingness signal" without needing to impute the actual year.

### Interaction Features

**Interaction features** capture the joint effect of two variables. For instance, the price premium for an extra room may depend on the apartment's total size, an additional room in a 200 sqm apartment is different from an additional room in a 50 sqm studio. We create the following interactions:

- `rooms_area = rooms * areaSqm` — total "room-area product", approximating how spacious each room is.
- `bath_area = bathrooms * areaSqm` — similar logic for bathrooms.
- `floor_elevator = floor * has_elevator` — a high floor *with* an elevator is more desirable than a high floor *without* one.
- `floor_view = floor * has_view` — similarly, a view from a high floor typically commands a bigger premium.

For the last two interactions, we first convert the boolean `elevator` and `view` columns from strings (`'True'`/`'False'`) to integers (1/0) so they can be used in arithmetic operations.


In [5]:
# Create binary indicator for renovationYear column
df['is_renovated'] = np.where(df['renovationYear'].isnull(), 0, 1)
# Create interaction features
df['rooms_area'] = df['rooms'] * df['areaSqm']
df['bath_area'] = df['bathrooms'] * df['areaSqm']
has_elevator = np.where(df['elevator'] == 'False', 0, 1)
has_view = np.where(df['view'] == 'False', 0, 1)
df['floor_elevator'] = df['floor'] * has_elevator
df['floor_view'] = df['floor'] * has_view


In [6]:
df.head()


,listingId,airConditioning,alarm,areaSqm,balcony,bathrooms,closedParkingSpots,elevator,energyClass,fireplace,...,yearBuilt,locationFloor,totalParkingSpots,apartmentAge,district,is_renovated,rooms_area,bath_area,floor_elevator,floor_view
0,20304396,True,False,3.850148,False,1.0,0.0,True,C,False,...,1975.0,Κέντρο Θεσσαλονίκης,0.0,51.0,1.0,1,3.850148,3.850148,7,7
1,19956518,True,False,3.218876,False,1.0,0.0,True,B,False,...,1982.0,Κέντρο Θεσσαλονίκης,0.0,44.0,1.0,1,3.218876,3.218876,8,8
2,18383741,False,False,4.094345,False,1.0,0.0,True,D,False,...,1978.0,Ανάληψη - Μπότσαρη - Νέα Παραλία,0.0,48.0,5.0,0,4.094345,4.094345,6,6
3,20242029,True,False,4.094345,False,1.0,0.0,True,F,False,...,1978.0,Ανάληψη - Μπότσαρη - Νέα Παραλία,0.0,48.0,5.0,0,4.094345,4.094345,8,8
4,20055845,True,False,4.317488,False,1.0,0.0,True,C,False,...,1981.0,Τούμπα,0.0,45.0,4.0,0,8.634976,4.317488,4,4


In [7]:
df.to_csv('spitogatos_feat_eng.csv', index=False)
